# 02 - Data Collection (Bước 2)
**Project:** Student Performance Analysis
**Branch:** `feature/data`
**Vai trò:** Data Engineer / Data Analyst

Mục tiêu: Thực hiện Bước 2 của đề tài: ghi nhận nguồn dữ liệu, khảo sát số bản ghi/thuộc tính/cấu trúc, lập từ điển dữ liệu, đối chiếu với các đầu vào đề tài yêu cầu, lưu trữ vào cơ sở dữ liệu và truy vấn bằng SQL, trước khi làm Data Cleaning.

**Đầu vào:** `data/raw/StudentPerformanceFactors.csv`
**Đầu ra:** `data/processed/raw_file_manifest.csv`, `data/processed/data_dictionary.csv`, `data/processed/topic_feature_coverage.csv`
**Công cụ:** Python/Pandas (đọc và kiểm tra), MySQL (lưu trữ), SQL (truy vấn và khai thác)

In [1]:
import hashlib
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_DIR / 'StudentPerformanceFactors.csv')
print('Shape:', df.shape)

Shape: (6607, 20)


## 1. Thông tin nguồn dữ liệu

Nguồn và định dạng lấy theo Topic (Kaggle, CSV). Các mục ghi `CẦN ĐIỀN` là thông tin file CSV không chứa (URL, tác giả, giấy phép, ngày tải), cần người thu thập bổ sung.

In [2]:
collection_info = {
    'Tên dataset': 'Student Performance Factors',
    'Chủ đề': 'Các yếu tố ảnh hưởng đến điểm thi của học sinh',
    'Đơn vị quan sát': '1 dòng = 1 học sinh (không có cột ID)',
    'Biến mục tiêu': 'Exam_Score',
    'Định dạng file': 'CSV',
    'Nguồn dữ liệu': 'Kaggle - Student Performance Factors (theo Topic, Bước 2)',
    'URL nguồn': 'CẦN ĐIỀN',
    'Chủ sở hữu / tác giả': 'CẦN ĐIỀN',
    'Giấy phép (license)': 'CẦN ĐIỀN',
    'Ngày thu thập / tải về': 'CẦN ĐIỀN',
    'Phương pháp thu thập': 'Tải file CSV từ Kaggle (theo Topic, Bước 2)',
}

pd.DataFrame(collection_info.items(), columns=['Thuộc tính', 'Giá trị'])

,Thuộc tính,Giá trị
0,Tên dataset,Student Performance Factors
1,Chủ đề,Các yếu tố ảnh hưởng đến điểm thi của học sinh
2,Đơn vị quan sát,1 dòng = 1 học sinh (không có cột ID)
3,Biến mục tiêu,Exam_Score
4,Định dạng file,CSV
5,Nguồn dữ liệu,"Kaggle - Student Performance Factors (theo Topic, Bước 2)"
6,URL nguồn,CẦN ĐIỀN
7,Chủ sở hữu / tác giả,CẦN ĐIỀN
8,Giấy phép (license),CẦN ĐIỀN
9,Ngày thu thập / tải về,CẦN ĐIỀN


## 2. Kiểm kê file dữ liệu thô

Ghi lại kích thước, số dòng, số cột và mã băm MD5 của từng file. Nếu sau này file trong `data/raw/` bị chỉnh sửa, mã MD5 sẽ khác và ta phát hiện được.

In [3]:
def md5_of(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

manifest_rows = []
for path in sorted(RAW_DIR.glob('*.csv')):
    d = pd.read_csv(path)
    manifest_rows.append({
        'file': path.name,
        'size_kb': round(path.stat().st_size / 1024, 1),
        'n_records': d.shape[0],
        'n_features': d.shape[1],
        'md5': md5_of(path),
    })

raw_file_manifest = pd.DataFrame(manifest_rows)
raw_file_manifest

,file,size_kb,n_records,n_features,md5
0,StudentPerformanceFactors.csv,626.9,6607,20,b4a4c6ce906010dc03956046fa8de891


## 3. Từ điển dữ liệu (Data Dictionary)

Mô tả và đơn vị của từng cột theo mô tả công khai của dataset Student Performance Factors. Nếu dữ liệu của bạn có định nghĩa khác (ví dụ đơn vị `Tutoring_Sessions`), sửa trực tiếp trong cell bên dưới.

In [4]:
# (column, group, kind, description, unit, allowed_values, min, max)
LMH = ['Low', 'Medium', 'High']
YN = ['Yes', 'No']
INF = np.inf

dict_rows = [
    ('Hours_Studied', 'STUDY BEHAVIOR', 'numeric', 'Số giờ học mỗi tuần', 'giờ/tuần', None, 0, 168),
    ('Attendance', 'STUDY BEHAVIOR', 'numeric', 'Tỷ lệ đi học', '%', None, 0, 100),
    ('Previous_Scores', 'STUDY BEHAVIOR', 'numeric', 'Điểm các kỳ thi trước', 'điểm (0-100)', None, 0, 100),
    ('Tutoring_Sessions', 'STUDY BEHAVIOR', 'numeric', 'Số buổi học kèm', 'buổi/tháng', None, 0, INF),
    ('Motivation_Level', 'STUDY BEHAVIOR', 'categorical', 'Mức độ động lực học tập', 'Low/Medium/High', LMH, None, None),
    ('Parental_Involvement', 'FAMILY', 'categorical', 'Mức độ tham gia của phụ huynh', 'Low/Medium/High', LMH, None, None),
    ('Family_Income', 'FAMILY', 'categorical', 'Mức thu nhập gia đình', 'Low/Medium/High', LMH, None, None),
    ('Parental_Education_Level', 'FAMILY', 'categorical', 'Trình độ học vấn cao nhất của phụ huynh', 'High School/College/Postgraduate',
     ['High School', 'College', 'Postgraduate'], None, None),
    ('School_Type', 'SCHOOL', 'categorical', 'Loại trường', 'Public/Private', ['Public', 'Private'], None, None),
    ('Teacher_Quality', 'SCHOOL', 'categorical', 'Chất lượng giáo viên', 'Low/Medium/High', LMH, None, None),
    ('Access_to_Resources', 'SCHOOL', 'categorical', 'Mức độ tiếp cận tài nguyên học tập', 'Low/Medium/High', LMH, None, None),
    ('Peer_Influence', 'SCHOOL', 'categorical', 'Ảnh hưởng của bạn bè', 'Positive/Neutral/Negative',
     ['Positive', 'Neutral', 'Negative'], None, None),
    ('Sleep_Hours', 'HEALTH & LIFESTYLE', 'numeric', 'Số giờ ngủ mỗi đêm', 'giờ/đêm', None, 0, 24),
    ('Physical_Activity', 'HEALTH & LIFESTYLE', 'numeric', 'Số giờ vận động thể chất mỗi tuần', 'giờ/tuần', None, 0, 168),
    ('Extracurricular_Activities', 'HEALTH & LIFESTYLE', 'categorical', 'Có tham gia hoạt động ngoại khóa', 'Yes/No', YN, None, None),
    ('Learning_Disabilities', 'HEALTH & LIFESTYLE', 'categorical', 'Có khó khăn trong học tập', 'Yes/No', YN, None, None),
    ('Internet_Access', 'ENVIRONMENT', 'categorical', 'Có truy cập Internet', 'Yes/No', YN, None, None),
    ('Distance_from_Home', 'ENVIRONMENT', 'categorical', 'Khoảng cách từ nhà đến trường', 'Near/Moderate/Far',
     ['Near', 'Moderate', 'Far'], None, None),
    ('Gender', 'DEMOGRAPHIC', 'categorical', 'Giới tính', 'Male/Female', ['Male', 'Female'], None, None),
    ('Exam_Score', 'TARGET', 'numeric', 'Điểm thi cuối kỳ (biến mục tiêu)', 'điểm (0-100)', None, 0, 100),
]

data_dictionary = pd.DataFrame(
    dict_rows,
    columns=['column', 'group', 'kind', 'description', 'unit_or_values', 'allowed_values', 'min_valid', 'max_valid']
)

# Từ điển phải phủ đúng toàn bộ cột của dữ liệu
only_in_data = set(df.columns) - set(data_dictionary['column'])
only_in_dict = set(data_dictionary['column']) - set(df.columns)
print('Cột có trong data nhưng thiếu trong từ điển:', sorted(only_in_data) or 'không')
print('Cột có trong từ điển nhưng không có trong data:', sorted(only_in_dict) or 'không')
print()
data_dictionary[['column', 'group', 'kind', 'description', 'unit_or_values']]

Cột có trong data nhưng thiếu trong từ điển: không
Cột có trong từ điển nhưng không có trong data: không



,column,group,kind,description,unit_or_values
0,Hours_Studied,STUDY BEHAVIOR,numeric,Số giờ học mỗi tuần,giờ/tuần
1,Attendance,STUDY BEHAVIOR,numeric,Tỷ lệ đi học,%
2,Previous_Scores,STUDY BEHAVIOR,numeric,Điểm các kỳ thi trước,điểm (0-100)
3,Tutoring_Sessions,STUDY BEHAVIOR,numeric,Số buổi học kèm,buổi/tháng
4,Motivation_Level,STUDY BEHAVIOR,categorical,Mức độ động lực học tập,Low/Medium/High
5,Parental_Involvement,FAMILY,categorical,Mức độ tham gia của phụ huynh,Low/Medium/High
6,Family_Income,FAMILY,categorical,Mức thu nhập gia đình,Low/Medium/High
7,Parental_Education_Level,FAMILY,categorical,Trình độ học vấn cao nhất của phụ huynh,High School/College/Postgraduate
8,School_Type,SCHOOL,categorical,Loại trường,Public/Private
9,Teacher_Quality,SCHOOL,categorical,Chất lượng giáo viên,Low/Medium/High


## 4. Kiểm tra schema

Đối chiếu dữ liệu thực tế với từ điển: kiểu dữ liệu, giá trị category hợp lệ, khoảng giá trị hợp lệ. Cột `n_invalid` là số dòng vi phạm (không tính giá trị thiếu, phần thiếu xử lý ở notebook 01 và 03).

In [5]:
check_rows = []
for col, group, kind, desc, unit, allowed, vmin, vmax in dict_rows:
    s = df[col]
    if kind == 'numeric':
        type_ok = pd.api.types.is_numeric_dtype(s)
        n_invalid = int(((s < vmin) | (s > vmax)).sum()) if type_ok else len(s)
        rule = f'{vmin} <= x <= {vmax}'
    else:
        type_ok = not pd.api.types.is_numeric_dtype(s)
        n_invalid = int((~s.dropna().isin(allowed)).sum())
        rule = 'trong ' + '/'.join(allowed)
    check_rows.append({
        'column': col,
        'type_ok': type_ok,
        'rule': rule,
        'n_missing': int(s.isna().sum()),
        'n_invalid': n_invalid,
        'status': 'OK' if (type_ok and n_invalid == 0) else 'KIỂM TRA',
    })

schema_check = pd.DataFrame(check_rows)
schema_check

,column,type_ok,rule,n_missing,n_invalid,status
0,Hours_Studied,True,0 <= x <= 168,0,0,OK
1,Attendance,True,0 <= x <= 100,0,0,OK
2,Previous_Scores,True,0 <= x <= 100,0,0,OK
3,Tutoring_Sessions,True,0 <= x <= inf,0,0,OK
4,Motivation_Level,True,trong Low/Medium/High,0,0,OK
5,Parental_Involvement,True,trong Low/Medium/High,0,0,OK
6,Family_Income,True,trong Low/Medium/High,0,0,OK
7,Parental_Education_Level,True,trong High School/College/Postgraduate,90,0,OK
8,School_Type,True,trong Public/Private,0,0,OK
9,Teacher_Quality,True,trong Low/Medium/High,78,0,OK


In [6]:
# Chi tiết các cột vi phạm schema
bad_cols = schema_check.loc[schema_check['status'] != 'OK', 'column'].tolist()
print('Cột vi phạm schema:', bad_cols or 'không')

for col in bad_cols:
    row = data_dictionary.set_index('column').loc[col]
    if row['kind'] == 'numeric':
        mask = (df[col] < row['min_valid']) | (df[col] > row['max_valid'])
        print(f"\n{col}: các dòng vi phạm")
        display(df.loc[mask, ['Hours_Studied', 'Attendance', 'Previous_Scores', col]])

Cột vi phạm schema: ['Exam_Score']

Exam_Score: các dòng vi phạm


,Hours_Studied,Attendance,Previous_Scores,Exam_Score
1525,27,98,93,101


## 5. Đối chiếu đầu vào của đề tài với dataset

Topic (Bước 1) yêu cầu 12 đầu vào cho mô hình. Cell dưới đối chiếu từng đầu vào với cột thực tế trong file CSV.

In [7]:
topic_inputs = [
    # (đầu vào theo Topic, cột trong dataset hoặc None, mức khớp, ghi chú)
    ('Study Hours',           'Hours_Studied',     'Đầy đủ',  'giờ học/tuần'),
    ('Attendance',            'Attendance',        'Đầy đủ',  '% đi học'),
    ('Previous Score',        'Previous_Scores',   'Đầy đủ',  ''),
    ('Assignment Completion', None,                'Thiếu',   'Không có cột tương đương'),
    ('Sleep Hours',           'Sleep_Hours',       'Đầy đủ',  ''),
    ('Exercise',              'Physical_Activity', 'Đầy đủ',  'giờ vận động/tuần'),
    ('Social Media',          None,                'Thiếu',   'Không có cột tương đương'),
    ('Gaming',                None,                'Thiếu',   'Không có cột tương đương'),
    ('Internet Usage',        'Internet_Access',   'Một phần', 'Chỉ có Yes/No, không có thời lượng sử dụng'),
    ('Stress',                None,                'Thiếu',   'Không có cột tương đương'),
    ('Motivation',            'Motivation_Level',  'Đầy đủ',  'Low/Medium/High'),
    ('Part-time Job',         None,                'Thiếu',   'Không có cột tương đương'),
    ('Exam Score (target)',   'Exam_Score',        'Đầy đủ',  'Biến mục tiêu'),
]

topic_feature_coverage = pd.DataFrame(topic_inputs, columns=['topic_input', 'dataset_column', 'coverage', 'note'])
# Kiểm tra cột được ghi nhận có thật sự tồn tại trong dữ liệu
mapped = topic_feature_coverage['dataset_column'].dropna()
assert mapped.isin(df.columns).all(), 'Có cột được ghi nhận nhưng không tồn tại trong data'

display(topic_feature_coverage)
print()
print(topic_feature_coverage['coverage'].value_counts().to_string())

# Cột có trong dataset nhưng Topic không nhắc tới
extra_cols = [c for c in df.columns if c not in set(mapped)]
print()
print('Cột có trong dataset nhưng không nằm trong danh sách đầu vào của Topic:')
print(extra_cols)

,topic_input,dataset_column,coverage,note
0,Study Hours,Hours_Studied,Đầy đủ,giờ học/tuần
1,Attendance,Attendance,Đầy đủ,% đi học
2,Previous Score,Previous_Scores,Đầy đủ,
3,Assignment Completion,NaN,Thiếu,Không có cột tương đương
4,Sleep Hours,Sleep_Hours,Đầy đủ,
5,Exercise,Physical_Activity,Đầy đủ,giờ vận động/tuần
6,Social Media,NaN,Thiếu,Không có cột tương đương
7,Gaming,NaN,Thiếu,Không có cột tương đương
8,Internet Usage,Internet_Access,Một phần,"Chỉ có Yes/No, không có thời lượng sử dụng"
9,Stress,NaN,Thiếu,Không có cột tương đương



coverage
Đầy đủ      7
Thiếu       5
Một phần    1

Cột có trong dataset nhưng không nằm trong danh sách đầu vào của Topic:
['Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Tutoring_Sessions', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']


## 6. Lưu trữ và truy vấn dữ liệu (MySQL + SQL)

Topic yêu cầu lưu trữ bằng MySQL và khai thác bằng SQL. Cell kết nối đọc chuỗi kết nối từ biến môi trường `MYSQL_URL` (không ghi mật khẩu trong notebook):

```
mysql+pymysql://<user>:<password>@localhost:3306/<database>
```

Nếu không đặt `MYSQL_URL`, notebook dùng SQLite trong bộ nhớ để vẫn chạy được. Các câu SQL bên dưới là SQL chuẩn, chạy giống nhau trên cả hai. **Bản notebook đã chạy sẵn dùng SQLite; cần chạy lại với `MYSQL_URL` để xác nhận trên MySQL.**

In [8]:
import os
import sqlite3

TABLE = 'student_performance'
MYSQL_URL = os.getenv('MYSQL_URL')

if MYSQL_URL:
    from sqlalchemy import create_engine
    conn = create_engine(MYSQL_URL)
    DB_ENGINE = 'MySQL'
else:
    conn = sqlite3.connect(':memory:')
    DB_ENGINE = 'SQLite (in-memory, dự phòng khi chưa đặt MYSQL_URL)'

# Ghi bảng thô vào cơ sở dữ liệu
df.to_sql(TABLE, conn, if_exists='replace', index=False)

def run_sql(query):
    return pd.read_sql_query(query, conn)

print('Cơ sở dữ liệu đang dùng:', DB_ENGINE)

# Kiểm tra đọc ngược lại khớp với file CSV
n_db = run_sql(f'SELECT COUNT(*) AS n FROM {TABLE}')['n'].iloc[0]
print(f'Số dòng trong DB: {n_db} | số dòng trong CSV: {len(df)} | khớp: {n_db == len(df)}')

Cơ sở dữ liệu đang dùng: SQLite (in-memory, dự phòng khi chưa đặt MYSQL_URL)
Số dòng trong DB: 6607 | số dòng trong CSV: 6607 | khớp: True


In [9]:
# Q1. Số học sinh và điểm trung bình theo loại trường
run_sql(f'''
SELECT School_Type,
       COUNT(*)                    AS n_students,
       ROUND(AVG(Exam_Score), 2)   AS avg_score
FROM {TABLE}
GROUP BY School_Type
ORDER BY n_students DESC
''')

,School_Type,n_students,avg_score
0,Public,4598,67.21
1,Private,2009,67.29


In [10]:
# Q2. Điểm thi trung bình theo nhóm chuyên cần
run_sql(f'''
SELECT CASE WHEN Attendance < 70 THEN '1. Duoi 70%'
            WHEN Attendance < 85 THEN '2. Tu 70% den 84%'
            ELSE '3. Tu 85% tro len' END AS attendance_band,
       COUNT(*)                  AS n_students,
       ROUND(AVG(Exam_Score), 2) AS avg_score
FROM {TABLE}
GROUP BY attendance_band
ORDER BY attendance_band
''')

,attendance_band,n_students,avg_score
0,1. Duoi 70%,1573,64.21
1,2. Tu 70% den 84%,2523,66.68
2,3. Tu 85% tro len,2511,69.69


In [11]:
# Q3. Đếm giá trị thiếu theo cột bằng SQL
run_sql(f'''
SELECT SUM(CASE WHEN Teacher_Quality IS NULL THEN 1 ELSE 0 END)           AS missing_teacher_quality,
       SUM(CASE WHEN Parental_Education_Level IS NULL THEN 1 ELSE 0 END)  AS missing_parental_education,
       SUM(CASE WHEN Distance_from_Home IS NULL THEN 1 ELSE 0 END)        AS missing_distance
FROM {TABLE}
''')

,missing_teacher_quality,missing_parental_education,missing_distance
0,78,90,67


In [12]:
# Q4. Kiểm tra dữ liệu bất thường bằng SQL (điểm thi ngoài thang 0-100)
run_sql(f'''
SELECT Hours_Studied, Attendance, Previous_Scores, Exam_Score
FROM {TABLE}
WHERE Exam_Score > 100 OR Exam_Score < 0
''')

,Hours_Studied,Attendance,Previous_Scores,Exam_Score
0,27,98,93,101


In [13]:
# Q5. Nhóm học sinh có dấu hiệu rủi ro: chuyên cần thấp và điểm cũ thấp
run_sql(f'''
SELECT COUNT(*)                  AS n_students,
       ROUND(AVG(Exam_Score), 2) AS avg_score,
       MIN(Exam_Score)           AS min_score
FROM {TABLE}
WHERE Attendance < 70 AND Previous_Scores < 65
''')

,n_students,avg_score,min_score
0,428,63.46,58


## 7. Lưu output

In [14]:
raw_file_manifest.to_csv(PROCESSED_DIR / 'raw_file_manifest.csv', index=False)

dd_out = data_dictionary.copy()
dd_out['allowed_values'] = dd_out['allowed_values'].apply(lambda v: '|'.join(v) if isinstance(v, list) else '')
dd_out.to_csv(PROCESSED_DIR / 'data_dictionary.csv', index=False)

topic_feature_coverage.to_csv(PROCESSED_DIR / 'topic_feature_coverage.csv', index=False)

print('Đã lưu:', sorted(p.name for p in PROCESSED_DIR.glob('*.csv')))

Đã lưu: ['data_dictionary.csv', 'raw_file_manifest.csv', 'topic_feature_coverage.csv']


## 8. Đối chiếu yêu cầu Bước 2 trong Topic

| Yêu cầu của Topic | Trạng thái | Nơi thực hiện |
|---|---|---|
| Nguồn dữ liệu: Kaggle, định dạng CSV | Đã ghi nhận nguồn và định dạng. Còn thiếu URL, tác giả, giấy phép, ngày tải | Mục 1 |
| Khảo sát số bản ghi, thuộc tính, cấu trúc | Đã làm: 6.607 bản ghi, 20 thuộc tính, kiểu dữ liệu, MD5 | Mục 2, 3, 4 |
| Mô tả nhóm dữ liệu (học tập, sinh hoạt, công nghệ, hành vi) | Đã làm qua từ điển dữ liệu. Nhóm công nghệ và hành vi chỉ có 2 cột (`Internet_Access`, `Motivation_Level`) | Mục 3 |
| Python/Pandas đọc và kiểm tra dữ liệu | Đã làm | Toàn bộ notebook |
| MySQL lưu trữ và quản lý | Mã đã viết, bản chạy sẵn dùng SQLite. Cần chạy lại với `MYSQL_URL` để xác nhận trên MySQL | Mục 6 |
| SQL truy vấn và khai thác dữ liệu | Đã làm 5 truy vấn (gom nhóm, CASE WHEN, đếm thiếu, kiểm tra bất thường) | Mục 6 |

**Vấn đề cần quyết định:** trong 13 mục đối chiếu (12 đầu vào và biến mục tiêu), dataset khớp đầy đủ 7 mục (6 đầu vào và `Exam_Score`), khớp một phần 1 mục (`Internet Usage`), và thiếu hẳn 5 mục: `Assignment Completion`, `Social Media`, `Gaming`, `Stress`, `Part-time Job`. Hệ quả: không tạo được đặc trưng `Digital Usage` ở Bước 4 và không phân tích được Social Media, Gaming, Stress với `Exam Score` như Topic mô tả. Hướng xử lý: (a) bổ sung dataset Student Habits (Kaggle) rồi ghép; hoặc (b) giữ dataset hiện tại và điều chỉnh phạm vi đề tài.